In [13]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression


class SimpleBaseballWinModel:
    """
    입력:
        home_team_code, away_team_code
    출력:
        홈팀 승률 (0~1)

    사용 feature:
        1) h2h_home_winrate   : 홈팀 vs 어웨이팀 상대전적 승률
        2) team_winrate_diff  : 홈팀 전체 승률 - 어웨이팀 전체 승률
        3) elo_diff           : 홈팀 Elo - 어웨이팀 Elo
    """

    def __init__(
        self,
        base_elo: float = 1500.0,
        k_factor: float = 20.0,
        reg_C: float = 1.0,
        min_games_prior: int = 0,
    ):
        self.base_elo = base_elo
        self.k_factor = k_factor
        self.reg_C = reg_C
        self.min_games_prior = min_games_prior

        self.model = LogisticRegression(C=self.reg_C, max_iter=1000)
        self.fitted = False

        # 학습 후 예측용 상태 저장
        self.latest_h2h = {}
        self.latest_team_record = {}
        self.latest_elo = {}

        self.feature_cols = [
            "h2h_home_winrate",
            "team_winrate_diff",
            "elo_diff",
        ]

    # --------------------------------------------------
    # 1. 데이터 로드
    # --------------------------------------------------
    @staticmethod
    def load_data(file_paths):
        dfs = [pd.read_csv(fp) for fp in file_paths]
        df = pd.concat(dfs, ignore_index=True)

        # 날짜 정렬
        df["game_date"] = pd.to_datetime(df["game_date"])
        df = df.sort_values(["game_date", "game_id"]).reset_index(drop=True)

        # 타깃 통일
        if "home_win" not in df.columns:
            if "y_home_win" in df.columns:
                df["home_win"] = df["y_home_win"]
            else:
                raise ValueError("home_win 또는 y_home_win 컬럼이 필요합니다.")

        required_cols = ["home_team_code", "away_team_code", "home_win", "game_date", "game_id"]
        missing = [c for c in required_cols if c not in df.columns]
        if missing:
            raise ValueError(f"필수 컬럼 누락: {missing}")

        df["home_win"] = df["home_win"].astype(int)
        return df

    # --------------------------------------------------
    # 2. leakage-free feature 생성
    # --------------------------------------------------
    def build_training_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        각 경기 시점 이전 정보만 사용해서 feature 생성
        """
        # 상대전적: (team_a, team_b) -> [wins_for_team_a, games]
        h2h = {}

        # 팀 전체 기록: team -> [wins, games]
        team_record = {}

        # Elo: team -> rating
        elo = {}

        feature_rows = []

        for _, row in df.iterrows():
            home = row["home_team_code"]
            away = row["away_team_code"]
            y = row["home_win"]

            # ---------- 현재 경기 이전 feature 계산 ----------
            # 1) 상대전적
            key = (home, away)
            if key in h2h and h2h[key][1] > 0:
                h2h_home_winrate = h2h[key][0] / h2h[key][1]
            else:
                h2h_home_winrate = 0.5

            # 2) 팀 전체 승률
            home_wins, home_games = team_record.get(home, [0, 0])
            away_wins, away_games = team_record.get(away, [0, 0])

            home_team_winrate = home_wins / home_games if home_games > 0 else 0.5
            away_team_winrate = away_wins / away_games if away_games > 0 else 0.5
            team_winrate_diff = home_team_winrate - away_team_winrate

            # 3) Elo
            home_elo = elo.get(home, self.base_elo)
            away_elo = elo.get(away, self.base_elo)
            elo_diff = home_elo - away_elo

            feature_rows.append(
                {
                    "game_id": row["game_id"],
                    "game_date": row["game_date"],
                    "home_team_code": home,
                    "away_team_code": away,
                    "home_win": y,
                    "h2h_home_winrate": h2h_home_winrate,
                    "team_winrate_diff": team_winrate_diff,
                    "elo_diff": elo_diff,
                    "home_team_games_before": home_games,
                    "away_team_games_before": away_games,
                }
            )

            # ---------- 경기 결과 반영 후 상태 업데이트 ----------
            # 상대전적 업데이트
            if (home, away) not in h2h:
                h2h[(home, away)] = [0, 0]
            if (away, home) not in h2h:
                h2h[(away, home)] = [0, 0]

            h2h[(home, away)][1] += 1
            h2h[(home, away)][0] += y

            h2h[(away, home)][1] += 1
            h2h[(away, home)][0] += (1 - y)

            # 팀 전체 기록 업데이트
            if home not in team_record:
                team_record[home] = [0, 0]
            if away not in team_record:
                team_record[away] = [0, 0]

            team_record[home][1] += 1
            team_record[away][1] += 1

            team_record[home][0] += y
            team_record[away][0] += (1 - y)

            # Elo 업데이트
            expected_home = 1.0 / (1.0 + 10 ** ((away_elo - home_elo) / 400.0))
            expected_away = 1.0 - expected_home

            new_home_elo = home_elo + self.k_factor * (y - expected_home)
            new_away_elo = away_elo + self.k_factor * ((1 - y) - expected_away)

            elo[home] = new_home_elo
            elo[away] = new_away_elo

        feat_df = pd.DataFrame(feature_rows)

        # 초반 경기 너무 불안하면 제외할 수 있게 옵션 제공
        if self.min_games_prior > 0:
            feat_df = feat_df[
                (feat_df["home_team_games_before"] >= self.min_games_prior)
                & (feat_df["away_team_games_before"] >= self.min_games_prior)
            ].reset_index(drop=True)

        # 예측용 최신 상태 저장
        self.latest_h2h = h2h
        self.latest_team_record = team_record
        self.latest_elo = elo

        return feat_df

    # --------------------------------------------------
    # 3. 모델 학습
    # --------------------------------------------------
    def fit(self, df: pd.DataFrame):
        feat_df = self.build_training_features(df)

        X = feat_df[self.feature_cols]
        y = feat_df["home_win"]

        self.model.fit(X, y)
        self.fitted = True
        self.train_features_ = feat_df.copy()
        return self

    # --------------------------------------------------
    # 4. 현재 시점 feature 계산
    # --------------------------------------------------
    def _current_features(self, home_team_code, away_team_code):
        if not self.fitted:
            raise RuntimeError("먼저 fit()을 실행하세요.")

        # 상대전적
        h2h_key = (home_team_code, away_team_code)
        if h2h_key in self.latest_h2h and self.latest_h2h[h2h_key][1] > 0:
            h2h_home_winrate = self.latest_h2h[h2h_key][0] / self.latest_h2h[h2h_key][1]
        else:
            h2h_home_winrate = 0.5

        # 팀 전체 승률
        home_wins, home_games = self.latest_team_record.get(home_team_code, [0, 0])
        away_wins, away_games = self.latest_team_record.get(away_team_code, [0, 0])

        home_team_winrate = home_wins / home_games if home_games > 0 else 0.5
        away_team_winrate = away_wins / away_games if away_games > 0 else 0.5
        team_winrate_diff = home_team_winrate - away_team_winrate

        # Elo
        home_elo = self.latest_elo.get(home_team_code, self.base_elo)
        away_elo = self.latest_elo.get(away_team_code, self.base_elo)
        elo_diff = home_elo - away_elo

        feature_dict = {
            "h2h_home_winrate": h2h_home_winrate,
            "team_winrate_diff": team_winrate_diff,
            "elo_diff": elo_diff,
        }
        return pd.DataFrame([feature_dict])

    # --------------------------------------------------
    # 5. 홈팀 승률 예측
    # --------------------------------------------------
    def predict_home_win_prob(self, home_team_code, away_team_code):
        X_new = self._current_features(home_team_code, away_team_code)
        prob = self.model.predict_proba(X_new)[0, 1]
        return float(prob)

    # --------------------------------------------------
    # 6. 현재 feature도 같이 보고 싶을 때
    # --------------------------------------------------
    def predict_with_features(self, home_team_code, away_team_code):
        X_new = self._current_features(home_team_code, away_team_code)
        prob = self.model.predict_proba(X_new)[0, 1]
        result = X_new.copy()
        result["pred_home_win_prob"] = prob
        result["home_team_code"] = home_team_code
        result["away_team_code"] = away_team_code
        return result




In [24]:
# ==================================================
# 사용 예시
# ==================================================
if __name__ == "__main__":
    file_paths = [
        "model_df_2023.csv",
        "model_df_2024.csv",
        "model_df_2025.csv",
    ]

    model = SimpleBaseballWinModel(
        base_elo=1500.0,
        k_factor=20.0,
        reg_C=1.0,
        min_games_prior=0,
    )

    df = model.load_data(file_paths)
    model.fit(df)

    ## 1001, 2002, 3001, 5002, 6002, 7002, 9002, 10001, 11001, 12001
    ## 삼성, 기아,  롯데, 엘지, 두산,  한화, 슥  , 키움,   엔씨,  크트

    # 예시
    away_team_code = 9002
    home_team_code = 11001
    

    prob = model.predict_home_win_prob(home_team_code, away_team_code)
    print(f"홈팀({home_team_code}) 승률: {prob*100:.2f}")

    detail = model.predict_with_features(home_team_code, away_team_code)
    print(detail)

홈팀(11001) 승률: 49.71
   h2h_home_winrate  team_winrate_diff   elo_diff  pred_home_win_prob  \
0          0.571429          -0.019048 -19.053134            0.497119   

   home_team_code  away_team_code  
0           11001            9002  


In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression

# --------------------------------------
# 1. 데이터 불러오기
# --------------------------------------
df23 = pd.read_csv("model_df_2023.csv")
df24 = pd.read_csv("model_df_2024.csv")
df25 = pd.read_csv("model_df_2025.csv")

df = pd.concat([df23, df24, df25], ignore_index=True)


In [3]:
df.head()

,game_id,game_date,season,home_team_code,away_team_code,stadium_code,home_sp_id,away_sp_id,home_sp_name,away_sp_name,...,top5_OPS_diff,top5_OBP_diff,sp_ERA_diff,sp_FIP_diff,sp_WHIP_diff,sp_K9_diff,sp_BB9_diff,sp_HR9_diff,sp_KBB_diff,y_home_win
0,20230711,2023-03-23 04:00:00,2023,10001,1001,1003,14674,11404,김동혁,장필준,...,0.015810,0.014867,-0.59,-1.38480,-0.32,0.371,-1.467,-0.702,0.4166,0
1,20230712,2023-03-23 04:00:00,2023,12001,5002,2002,11318,14780,엄상백,강효종,...,0.000434,0.003559,-2.60,-1.42946,-0.55,1.358,-2.648,0.069,1.9023,0
2,20230716,2023-03-24 04:00:00,2023,10001,1001,1003,11379,14613,최원태,허윤동,...,0.024522,0.027390,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,20230717,2023-03-24 04:00:00,2023,12001,5002,2002,15542,13102,슐서,김영준,...,0.007521,-0.003020,-21.38,-8.81277,-4.39,6.342,-24.463,1.087,2.5000,1
4,20230718,2023-03-24 04:00:00,2023,7002,6002,4001,10610,13061,장민재,곽빈,...,-0.009236,-0.007775,1.93,0.87580,0.25,0.465,-1.229,1.070,0.9451,0


In [4]:
# --------------------------------------
# 2. 상대전적 feature 생성
# --------------------------------------
def build_head2head_features(df):
    records = {}

    h2h_winrate = []
    
    for idx, row in df.iterrows():
        h = row["home_team_code"]
        a = row["away_team_code"]
        key = (h, a)

        # 과거 상대전적
        if key in records:
            wins, games = records[key]
            win_rate = wins / games
        else:
            win_rate = 0.5  # 초기값

        h2h_winrate.append(win_rate)

        # 결과 업데이트
        result = row["home_win"]
        
        if key not in records:
            records[key] = [0, 0]
        if (a, h) not in records:
            records[(a, h)] = [0, 0]

        # 홈 기준 업데이트
        records[key][1] += 1
        records[key][0] += result

        # 반대도 같이 업데이트
        records[(a, h)][1] += 1
        records[(a, h)][0] += (1 - result)

    df["h2h_winrate"] = h2h_winrate
    return df

df = build_head2head_features(df)


In [7]:
df.head()

,game_id,game_date,season,home_team_code,away_team_code,stadium_code,home_sp_id,away_sp_id,home_sp_name,away_sp_name,...,top5_OBP_diff,sp_ERA_diff,sp_FIP_diff,sp_WHIP_diff,sp_K9_diff,sp_BB9_diff,sp_HR9_diff,sp_KBB_diff,y_home_win,h2h_winrate
0,20230711,2023-03-23 04:00:00,2023,10001,1001,1003,14674,11404,김동혁,장필준,...,0.014867,-0.59,-1.38480,-0.32,0.371,-1.467,-0.702,0.4166,0,0.5
1,20230712,2023-03-23 04:00:00,2023,12001,5002,2002,11318,14780,엄상백,강효종,...,0.003559,-2.60,-1.42946,-0.55,1.358,-2.648,0.069,1.9023,0,0.5
2,20230716,2023-03-24 04:00:00,2023,10001,1001,1003,11379,14613,최원태,허윤동,...,0.027390,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0
3,20230717,2023-03-24 04:00:00,2023,12001,5002,2002,15542,13102,슐서,김영준,...,-0.003020,-21.38,-8.81277,-4.39,6.342,-24.463,1.087,2.5000,1,0.0
4,20230718,2023-03-24 04:00:00,2023,7002,6002,4001,10610,13061,장민재,곽빈,...,-0.007775,1.93,0.87580,0.25,0.465,-1.229,1.070,0.9451,0,0.5


In [8]:
# --------------------------------------
# 3. 모델 학습
# --------------------------------------
X = df[["h2h_winrate"]]
y = df["home_win"]

model = LogisticRegression()
model.fit(X, y)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [10]:
# --------------------------------------
# 4. 예측 함수
# --------------------------------------
# 현재까지의 상대전적 dictionary 생성
def build_records(df):
    records = {}
    for _, row in df.iterrows():
        h = row["home_team_code"]
        a = row["away_team_code"]
        result = row["home_win"]

        key = (h, a)
        if key not in records:
            records[key] = [0, 0]

        records[key][1] += 1
        records[key][0] += result

    return records

records = build_records(df)


def predict_home_win_prob(home_id, away_id):
    key = (home_id, away_id)

    if key in records:
        wins, games = records[key]
        win_rate = wins / games
    else:
        win_rate = 0.5

    prob = model.predict_proba([[win_rate]])[0][1]
    return prob



In [12]:
# --------------------------------------
# 5. 사용 예시
# --------------------------------------
## 1001, 2002, 3001, 5002, 6002, 7002, 9002, 10001, 11001, 12001
## 삼성, 기아,  롯데, 엘지, 두산,  한화, 슥  , 키움,   엔씨,  크트


home_id = 5002
away_id = 3001

prob = predict_home_win_prob(home_id, away_id)
print(f"홈팀 승률: {prob:.3f}")

홈팀 승률: 0.529


c:\Users\User\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
